# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- Conduct EDA: visualization and statistical measures to systematically understand the structure of the data.
- Recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- Inspect NaNs, datatypes, and summary statistics

In [ ]:
#Load the data as aviation_df:
aviation_df = pd.read_csv("data/AviationData.csv", encoding = "cp1252")
aviation_df.head()

I can already see there are many differnet data types and NaNs... Let's see more info.

In [ ]:
aviation_df.info()

Oof, there's a lot of missing data in MANY columns... too many to even list here. Some columns look to have over 70,000 NaNs!

In [ ]:
aviation_df.isna()

Lots of missing data in the `Latitude`, `Longitude`, `Airport.Code`, `Airport.Name`, `Air.carrier`, and other columns.

In [ ]:
aviation_df.describe()

From just these statistics, it seems like the total number of fatal and serious injuries is 0 at least 75% of the time, but the max number is 349 fatalities, which is really bad. I can't tell yet if 349 is an outlier, but it looks good to notice that most of the time, there are 0 fatal injuries. Even serious and minor injuries are 0 at least 75% of the time.

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- Inspect relevant columns
- Figure out any reasonable imputations
- Filter the dataset

In [ ]:
#See what values occur in the "Aircraft.Category" column:
aviation_df["Aircraft.Category"].value_counts()

In [ ]:
#We only want rows with "Airplane" as the entry:
aviation_df = aviation_df[aviation_df["Aircraft.Category"] == "Airplane"]
#Check that it worked:
aviation_df["Aircraft.Category"]

In [ ]:
#See what values occur in the "Amateur.Built" column:
aviation_df["Amateur.Built"].value_counts()

In [ ]:
#We only want rows with "No" as the entry because we only want professional
#builds:
aviation_df = aviation_df[aviation_df["Amateur.Built"] == "No"]
#Check if it worked:
aviation_df["Amateur.Built"]

In [ ]:
#First, check what dtype the column "Event.Date" is:
aviation_df["Event.Date"].dtype

In [ ]:
#We don't want it to be an object, we want datetime, so let's convert it:
aviation_df["Event.Date"] = pd.to_datetime(aviation_df["Event.Date"])
#Let's check if that worked:
aviation_df["Event.Date"].dtype

In [ ]:
#Awesome, now we only want rows with dates from 1983 or later:
aviation_df = aviation_df[aviation_df["Event.Date"].dt.year >= 1983]
#Check if it worked:
aviation_df["Event.Date"]

### Cleaning and Constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct a metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious/fatal injury can be estimated as a fraction from this.

In [ ]:
#First, I need to see what columns are in the dataset:
aviation_df.columns

In [ ]:
#Maybe we can calculate how many total passengers were on each flight by adding
#together the "Total.Fatal.Injuries", "Total.Serious.Injuries",
#"Total.Minor.Injuries", and "Total.Uninjured" columns. Let's create a new
#column for this:
aviation_df["Total.Passengers"] = aviation_df[
    ["Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"]
    #Using .sum() ensures NaNs are skipped and axis = 1 ensures we are summing
    #across columns:
    ].sum(axis = 1)

#Check changes:
aviation_df.head()

In [ ]:
#For injured passengers of any kind, NaN probably means 0 injuries recorded, so
#let's replace NaNs in injury columns with 0s:
cols = [
    "Total.Passengers",
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]

aviation_df[cols] = aviation_df[cols].fillna(0)

In [ ]:
#Now let's calculate how many people on each flight had fatal or serious #injuries by calculating that fraction of total passengers on each flight:
aviation_df["Serious.or.Fatal.Fraction"] = (
    (aviation_df["Total.Fatal.Injuries"] +
     aviation_df["Total.Serious.Injuries"]) /
    aviation_df["Total.Passengers"]
)

#Check changes:
aviation_df.head()

In [ ]:
#Let's check if the "Serious.or.Fatal.Fraction" column has any NaNs (infinity
#after calculating that fraction):
print(aviation_df["Serious.or.Fatal.Fraction"].isna().sum())

In [ ]:
#That's a lot of NaNs. Let's see what rows are causing those NaNs:
aviation_df[aviation_df["Serious.or.Fatal.Fraction"].isna()][
["Total.Passengers", "Total.Fatal.Injuries", "Total.Serious.Injuries"]
].head()

Looks like the NaNs are caused by rows where there are 0 injuries, but also "Total.Passengers" is 0, and 0/0 = infinity = NaN. For the purpose of figuring out how many flights had serious or fatal injuries, 0 injuries / 0 people is stil a 0% injury rate. So I'm going to fill those NaNs in the `Serious.or.Fatal.Fraction` column with 0s.

In [ ]:
aviation_df["Serious.or.Fatal.Fraction"] = (
    aviation_df["Serious.or.Fatal.Fraction"].fillna(0))
#Let's check if it worked:
aviation_df["Serious.or.Fatal.Fraction"].isna().sum()

Awesome! Now we have a colum with the total fraction of passengers with serious or fatal injuries, and that column has no NaNs! We can see the first five lines of that column below:

In [ ]:
aviation_df["Serious.or.Fatal.Fraction"].head()

**Aircraft.Damage**
- Identify and execute any cleaning tasks.
- Construct a derived column tracking whether an aircraft was destroyed or not.

In [ ]:
#First, let's look at some info about the "Aircraft.damage" column:
aviation_df["Aircraft.damage"].head(15)

In [ ]:
#There are some NaNs here. Let's check the value_counts() to see if we have
#enough data for dropping:
aviation_df["Aircraft.damage"].value_counts()

In [ ]:
#We only REALLY care about aircraft that are destroyed, so I'm going to remove
#rows with NaNs in the "Aircraft.damage" column because we have enough
#representation from other categories with actual data:
aviation_df = aviation_df.dropna(subset=["Aircraft.damage"])
#Check if it worked:
aviation_df["Aircraft.damage"].isna().sum()

In [ ]:
#Now, let's create a derived column that tracks whether an aircraft was
#destroyed or not:
aviation_df["Aircraft.Destroyed"] = (
    aviation_df["Aircraft.damage"] == "Destroyed"
)
#Let's check our dataset so far:
aviation_df.head()

### Investigate the *Make* column
- Identify cleaning tasks here.
- List cleaning tasks clearly in markdown.
- Execute the cleaning tasks.
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50, though lower could work as well).

In [ ]:
aviation_df["Make"].info()

Ok, not a lot of NaNs. That's good. Let's look at part of the column, and then check the value_counts:

In [ ]:
aviation_df["Make"].head(15)

In [ ]:
aviation_df["Make"].value_counts().head(50)

Oof ok, there are a lot of data entry issues causing the same Make to be counted multiple times (e.g. "CESSNA" and "Cessna" should be the same.) Less fix that.

In [ ]:
#First, let's convert every entry in the column to uppercase and strip
#whitespace, which should help combine some of these columns:
aviation_df["Make"] = aviation_df["Make"].str.upper().str.strip()
#Now let's check the value_counts again:
aviation_df["Make"].value_counts().head(50)

Much better, but still some companies that should be combined (e.g. "AIR TRACTOR INC" and "AIR TRACTOR"). Let's fix this:

In [ ]:
aviation_df.loc[aviation_df["Make"].str.contains("CESSNA", na=False), "Make"] = "CESSNA"
aviation_df.loc[aviation_df["Make"].str.contains("PIPER", na=False), "Make"] = "PIPER"
aviation_df.loc[aviation_df["Make"].str.contains("BEECH", na=False), "Make"] = "BEECH"
aviation_df.loc[aviation_df["Make"].str.contains("BOEING", na=False), "Make"] = "BOEING"
aviation_df.loc[aviation_df["Make"].str.contains("MOONEY", na=False), "Make"] = "MOONEY"
aviation_df.loc[aviation_df["Make"].str.contains("AIR TRACTOR", na=False), "Make"] = "AIR TRACTOR"
aviation_df.loc[aviation_df["Make"].str.contains("GRUMMAN", na=False), "Make"] = "GRUMMAN"
aviation_df.loc[aviation_df["Make"].str.contains("AIRBUS", na=False), "Make"] = "AIRBUS"
aviation_df.loc[aviation_df["Make"].str.contains("ROCKWELL", na=False), "Make"] = "ROCKWELL"
aviation_df.loc[aviation_df["Make"].str.contains("AVIAT", na=False), "Make"] = "AVIAT"
aviation_df.loc[aviation_df["Make"].str.contains("CIRRUS", na=False), "Make"] = "CIRRUS"
aviation_df.loc[aviation_df["Make"].str.contains("AYRES", na=False), "Make"] = "AYRES"
aviation_df.loc[aviation_df["Make"].str.contains("DE HAVILLAND", na=False), "Make"] = "DEHAVILLAND"
aviation_df.loc[aviation_df["Make"].str.contains("FLIGHT DESIGN", na=False), "Make"] = "FLIGHT DESIGN"
#Now let's check the value_counts again:
aviation_df["Make"].value_counts().head(50)

Ok, now let's keep the Makes with more than 50 counts to look at later, and group everything else into "OTHER":

In [ ]:
make_counts = aviation_df["Make"].value_counts()
valid_makes = make_counts[make_counts > 50].index
aviation_df["Make"] = aviation_df["Make"].apply(
    lambda x: x if x in valid_makes else "OTHER"
)
#Check if it worked:
aviation_df["Make"].value_counts()

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [ ]:
#First, get rid of any NaNs in the "Model" column:
aviation_df = aviation_df.dropna(subset=["Model"])

Now, check if model names repeat across different airplane makers.
In other words, if I just say "737", does that automatically tell me that it's a Boeing, or do I need to say "Boeing 737"? Likely it's the latter, but let's check.

#### Here's how to read the output of the below code:

Model   → Number of Different Makes Who Use that Model Name

172     → 1

737     → 1

PA28    → 1

MODEL_X → 3 different makes

In [ ]:
aviation_df.groupby("Model")["Make"].nunique().sort_values(ascending = False).head(20)

Ok so clearly most model names are used by many different airplane makers, which makes sense. This means we need to create a derived column that is a unique identifier for each plane type (i.e. Boeing 737 is different from a Cessna 737).

In [ ]:
aviation_df["Make.Model"] = aviation_df["Make"] + " " + aviation_df["Model"]
aviation_df["Make.Model"].head()

### Cleaning Other Columns
There are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks.

**Hint**: *Some things you might want to think about are values in numerical features that do not make sense or categories that contain too few examples. Consider treating values (for either categorical or numeric data) that might be placeholders for NaNs.*

**Note**: You do not necessarily need to impute or drop NaNs here.

#### Inspecting and Cleaning `Engine.Type`

In [ ]:
#Insepct the value_counts of Engine.Type:
aviation_df["Engine.Type"].value_counts()

In [ ]:
#Data cleaning of Engine.Type:
aviation_df["Engine.Type"] = aviation_df["Engine.Type"].str.strip()
aviation_df["Engine.Type"] = aviation_df["Engine.Type"].replace("UNK", "Unknown")
#Check if it worked:
aviation_df["Engine.Type"].value_counts()

#### Inspecting and Cleaning `Weather.Condition`

In [ ]:
#Inspect the value_counts of Weather.Condition:
aviation_df["Weather.Condition"].value_counts()

In [ ]:
#Data cleaning of Weather.Condition:
aviation_df["Weather.Condition"] = aviation_df["Weather.Condition"].str.strip()
aviation_df["Weather.Condition"] = aviation_df["Weather.Condition"].replace("Unk", "UNK")
#Check if it worked:
aviation_df["Weather.Condition"].value_counts()

#### Inspecting and Cleaning `Number.of.Engines`

In [ ]:
#Inspect the value_counts of Number.of.Engines:
aviation_df["Number.of.Engines"].value_counts()

In [ ]:
#Data cleaning of Number.of.Engines:
#We can only have a Number.of.Engines greater than 0, so let's turn all zeros
#into NaNs so they don't mess up our data later:
aviation_df.loc[aviation_df["Number.of.Engines"] == 0, "Number.of.Engines"] = pd.NA
#Check if it worked:
aviation_df["Number.of.Engines"].value_counts()

#### Inspecting and Cleaning `Purpose.of.flight`

In [ ]:
#Inspect the value_counts of Purpose.of.flight:
aviation_df["Purpose.of.flight"].value_counts()

In [ ]:
#Data cleaning of Purpose.of.flight:
aviation_df["Purpose.of.flight"] = aviation_df["Purpose.of.flight"].str.title().str.strip()
#Clean duplicates:
aviation_df["Purpose.of.flight"] = aviation_df["Purpose.of.flight"].replace({
    "Air Race/show": "Air Race Show",
    "Air Race show": "Air Race Show",
    "Executive/corporate": "Executive/Corporate",
})
#Only keep flight purposes that occur over 100 times:
purpose_counts = aviation_df["Purpose.of.flight"].value_counts()
keep_purposes = purpose_counts[purpose_counts > 100].index
aviation_df["Purpose.of.flight"] = aviation_df["Purpose.of.flight"].apply(
    lambda x: x if x in keep_purposes else "Other"
)
#Check if it worked:
aviation_df["Purpose.of.flight"].value_counts()

#### Inspecting and Cleaning `Broad.phase.of.flight`

#Inspect the value_counts of Broad.phase.of.flight:
aviation_df["Broad.phase.of.flight"].value_counts()


In [ ]:
#Inspect the value_counts of Broad.phase.of.flight:
aviation_df["Broad.phase.of.flight"].value_counts()

In [ ]:
#Data cleaning of Broad.phase.of.flight:
aviation_df["Broad.phase.of.flight"] = aviation_df["Broad.phase.of.flight"].str.strip()
#Only keep flight phases that appear more than 50 times:
phase_counts = aviation_df["Broad.phase.of.flight"].value_counts()
keep_phases = phase_counts[phase_counts > 50].index
aviation_df["Broad.phase.of.flight"] = aviation_df["Broad.phase.of.flight"].apply(lambda x: x if x in keep_phases else "Other")
#Check if it worked:
aviation_df["Broad.phase.of.flight"].value_counts()

### Column Removal
Inspect the DataFrame and drop any columns that have too many NaNs. For this exercise, keep all columns with more than 20,000 non-nulls.

In [ ]:
#First, inspect the DataFrame to figure out how many non-null values are in
#each column:
aviation_df.info()

In [ ]:
#Now, drop any columns with less than 20,000 non-null values:
aviation_df = aviation_df.drop(columns = ["Latitude", "Longitude", "Airport.Code", "Airport.Name", "Injury.Severity", "FAR.Description", "Schedule", "Air.carrier", "Report.Status", "Publication.Date"])
#I kept Weather.Condition because the assignment told us it will be useful later.
#Now, for the last time, let's take a look at the dataset's head with what remains:
aviation_df.head()

### Save DataFrame to CSV
- It's generally useful to save data to file/server after it's in a sufficiently cleaned or intermediate state.
- The data can then be loaded directly in another notebook for further analysis.
- This helps keep your notebooks and workflow readable, clean and modularized.

In [ ]:
aviation_df.to_csv("data/AviationDataCleaned.csv", index = False)